# T10 — Per-neuron task selectivity

Task T10 from the `Aditya` section of the repo README: *"the paper's most memorable figure,
and it does not exist yet."* Unlike T6 this one **needs code**, so the notebook writes the
code, tests it, generates the checkpoints, and renders the figure.

**The question.** Are hidden units **task-shared** (one unit responds to many tasks) or
**task-specific** (each unit owns a few)? And does the radial penalty move the network between
those regimes? Note this is *not* about dead units: `dead_frac` is already logged every task in
every run and is **0.000** throughout — every unit stays alive across all 150 tasks. The
question is what the live units are doing.

## What gets built

1. `--save_final_checkpoint` in `src/train.py` — end-of-run `state_dict`, plus a `config.py`
   change (cell 5) that you must not skip.
2. `S10_selectivity` in `src/studies.py` — 4 conditions x 3 seeds = **12 runs**, checkpointed.
3. `analysis/selectivity.py` — per-unit, per-task mean absolute response, normalised per unit.
4. `fig_selectivity()` in `analysis/figures.py` — units x tasks heatmap, one panel per condition.
5. `tests/test_selectivity.py` — a regression test, as every other metric in this repo has.

Why re-run rather than reuse the S3 arms: **no checkpoint exists anywhere in the repo**
(`find . -name "*.pt"` returns nothing) and there never was code to save one, so every finished
run discarded its weights on exit. The README says to expect this.

## Runtime budget on a Kaggle T4

Same measured basis as T6 — 79 archived runs at this configuration, median **63.6 min** on the
lab GPU at 8-way concurrency. S10 leaves `track_drift` off (cheaper); a T4 with ~4 CPU cores is
dearer. Planning number: **~90 min per run** (75-120). The checkpoint write is a few MB and
costs nothing.

| | runs | waves at concurrency 2 | wall-clock |
|---|---|---|---|
| `SHARD = "baseline"` | 3 | 2 | **~3 h** |
| `SHARD = "baseline,penalty"` | 6 | 3 | **~4.5 h** |
| all four conditions | 12 | 6 | ~9 h |

Two sessions of two conditions is the efficient split; Kaggle allows 2 concurrent GPU sessions,
so both can run at once. T6 (~13.5 h) and T10 (~9 h) together fit the 30 h weekly quota.

Checkpoints are ~11 MB each (2.8 M params, fp32), ~130 MB for all 12 — trivial against the
20 GB output limit.

## Overnight checklist

1. **Internet ON**, **Accelerator GPU T4 x2**.
2. **Save & Run All (Commit)**, not interactive — an interactive session idles out.
3. **Sessions 2+:** Add Data -> Your Notebooks -> Output, attach every earlier session.

## What makes it failsafe

* **Hard time budget** — the launcher is SIGTERMed at `TIME_BUDGET_H` by a watchdog thread that
  runs independently of the child's output, so a hung run cannot outlast it. The notebook then
  continues so `/kaggle/working` is saved; a session killed at Kaggle's 12 h wall saves nothing.
* **Resumable** — completed runs are skipped; interrupted ones are re-run.
* **`SHARD = "auto"`** picks the condition with the most unfinished runs.
* **Idempotent patches** — the code cells detect their own prior application, so re-running the
  notebook on a restored session does not double-patch.
* **Nothing after the launch raises** — analysis cells catch their own exceptions.

## 1. Configuration

In [ ]:
# Which conditions to run this session.
#   "auto" -> the condition with the most unfinished runs (resolved in cell 8)
#   "baseline" | "penalty" | "limit_tangential" | "limit_ste" | "all"
#   comma-separated combinations allowed, e.g. "baseline,penalty"
SHARD = "all"

# Stop launching and shut down cleanly after this many hours so the notebook finishes and
# /kaggle/working is saved. MEASURED: with `--timeout 43200` a session reserves ~11.9 h of
# quota, so ~12 h is the real ceiling. A session killed at that ceiling never reaches the
# save cells and its output is LOST, so this stays under it.
TIME_BUDGET_H = 11.0

CONCURRENCY = 2      # CONCURRENCY * THREADS must stay <= nproc (metric SVDs are CPU-bound)
THREADS     = 2
GPUS        = "0,1"

REPO  = "https://github.com/frisco137/rs_for_optimisation.git"
WORK  = "/kaggle/working/rs_for_optimisation"
STUDY = "S10_selectivity"
ROOT  = "experiments/permuted_mnist/selectivity"
CONDITIONS = ["baseline", "penalty", "limit_tangential", "limit_ste"]

print(f"shard={SHARD}  budget={TIME_BUDGET_H}h  concurrency={CONCURRENCY}x{THREADS} threads")

## 2. Environment

Fatal on purpose: a run without a GPU would take days and produce nothing usable, and a
`CONCURRENCY x THREADS` above `nproc` makes the CPU-bound metric SVDs thrash — Round 1-4 saw
load average 484 and tasks failing to advance for four minutes.

In [ ]:
import multiprocessing, os, shutil, subprocess, sys, time

SESSION_START = time.time()

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
ncpu = multiprocessing.cpu_count()
print("CPU cores:", ncpu, "| free disk GB:", round(shutil.disk_usage("/kaggle/working").free / 1e9, 1))
assert CONCURRENCY * THREADS <= ncpu, f"{CONCURRENCY}x{THREADS} exceeds {ncpu} cores"

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
assert torch.cuda.is_available(), "Turn on the GPU accelerator (Settings -> Accelerator)."

# The GPU must actually be usable by THIS torch build. Kaggle hands out a P100 (sm_60) when
# the accelerator is left as plain "GPU", and its PyTorch ships no Pascal kernels -- every run
# then dies on its first CUDA op with "no kernel image is available for execution on the
# device", after the CPU-only checks above have all passed. Fail here, in seconds, instead.
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"device: {name}  sm_{cap[0]}{cap[1]}  | torch arches: {torch.cuda.get_arch_list()}")
try:
    _t = torch.zeros(64, 64, device="cuda")
    _ = (_t @ _t).sum().item()
    torch.cuda.synchronize()
except Exception as e:
    raise SystemExit(
        f"GPU UNUSABLE: {name} (sm_{cap[0]}{cap[1]}) with this torch build -- {e}\n"
        f"Set the accelerator to T4 x2: kernel-metadata.json \"machine_shape\": \"NvidiaTeslaT4\", "
        f"or Settings -> Accelerator -> GPU T4 x2 in the editor. Do not launch on a P100.")
print("GPU compute check passed")
del _t

## 3. Clone and install

In [ ]:
import os, shutil, subprocess, sys, time

# A transient git failure must not end the session: one observed clone returned exit 128 with
# a healthy network moments after an identical clone succeeded. Retry with backoff, and clear
# any half-written tree between attempts.
for attempt in range(1, 6):
    if os.path.exists(os.path.join(WORK, "src", "train.py")):
        break
    shutil.rmtree(WORK, ignore_errors=True)
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, WORK],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"cloned on attempt {attempt}")
        break
    print(f"clone attempt {attempt} failed (exit {r.returncode}): {r.stderr.strip()[-200:]}")
    time.sleep(15 * attempt)
else:
    raise SystemExit("CLONE FAILED after 5 attempts -- not launching.")

os.chdir(WORK)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip install -q -r requirements.txt

## 4. Preflight

`src/data/` — the package holding `PermutedMNIST` — was missing from the repository until
commit `bff5bd9`: `.gitignore` had `*/data/`, meant for the downloaded MNIST in `data/`, and
the pattern also matched `src/data/`. `src/train.py:35-36` imports it, so every run died at
import while the 30 tests still passed and `--dry-run` still printed its commands. It is
tracked now; this cell refuses to launch if that ever regresses.

`torchvision` is a hard dependency of the loader and is **not** in `requirements.txt`. Kaggle
preinstalls it; the cell installs it if absent rather than letting 18 runs die on start.

The loader check confirms the property the paired analysis depends on: two arms built with the
same seed get the same 150 permutations, so `baseline/seed_1` and `penalty/seed_1` differ only
by the penalty.

In [ ]:
import os, subprocess, sys

assert os.path.exists("src/data/permuted_mnist.py"), (
    "src/data/ is missing -- the .gitignore `*/data/` bug is back. "
    "Pull latest main (fixed in bff5bd9); do not launch without it.")

try:
    import torchvision
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchvision"], check=True)
    import torchvision
print("torchvision", torchvision.__version__)

chk = subprocess.run([sys.executable, "-c", "import src.train"],
                     env={**os.environ, "PYTHONPATH": "."}, capture_output=True, text=True)
if chk.returncode != 0:
    print(chk.stderr.strip()[-800:])
    raise SystemExit("PREFLIGHT FAILED -- src.train does not import; do not launch.")
print("preflight OK -- src.train imports")

sys.path.insert(0, ".")
import torch
from src.data.permuted_mnist import PermutedMNIST

a = PermutedMNIST(n_tasks=150, device="cpu", seed=1)
b = PermutedMNIST(n_tasks=150, device="cpu", seed=1)
c = PermutedMNIST(n_tasks=150, device="cpu", seed=2)
assert all(torch.equal(x, y) for x, y in zip(a.permutations, b.permutations)), \
    "same seed must give the same task sequence -- paired arms would not be paired"
assert not torch.equal(a.permutations[5], c.permutations[5]), "different seeds must differ"
assert len({tuple(p.tolist()) for p in a.permutations}) == 150, "permutations must be distinct"
print("loader OK -- seed 1 reproducible, seeds independent, 150 distinct permutations")

x = a.x_train
print(f"pixel mean {x.mean():+.4f}  std {x.std():.4f}   (standardised: 0.0 / 1.0)")
print(f"||x|| mean {x.norm(dim=1).mean():.2f}          (expected ~27.7)")
del a, b, c, x

## 4. Patch: `--save_final_checkpoint`

Three edits, and **the third one is the whole point of reading this cell.**

`config_hash()` in `src/config.py` hashes the entire argparse namespace as "the scientific
content of a run". Adding any new argparse flag therefore changes the hash of *every* run in
*every* study — the launcher would declare all existing completed runs `stale` and re-run
them, and `load_arm` would refuse to combine an old seed with a new one inside the same arm.

`save_final_checkpoint` is an I/O flag: it changes what gets written to disk, not what gets
computed. So it joins `num_threads` in the ignore set. Cell 5 verifies this actually held.

`SCHEMA_VERSION` is **not** bumped: no column is added to, removed from, or redefined in
`metrics.parquet`, and the version docstring is explicit that a column change is the trigger.

The third edit is `BOOL_FLAGS` in `src/launch.py`. A `store_true` flag not listed there gets
emitted as `--save_final_checkpoint True`, which argparse rejects and every run dies instantly.

In [ ]:
import pathlib

def patch(path, anchor, insert, marker):
    p = pathlib.Path(path); s = p.read_text()
    if marker in s:
        print(f"  {path}: already patched"); return
    assert s.count(anchor) == 1, f"anchor not unique in {path}: {s.count(anchor)} hits"
    p.write_text(s.replace(anchor, anchor + insert))
    print(f"  {path}: patched")

# (a) config.py -- keep the I/O flag out of the comparability hash
patch("src/config.py",
      '        "diverged_at_task", "num_threads",\n',
      '        "save_final_checkpoint",\n',
      marker='"save_final_checkpoint",')

# (b) train.py -- the argparse flag
patch("src/train.py",
      '    p.add_argument("--track_drift", action="store_true")\n',
      '    p.add_argument("--save_final_checkpoint", action="store_true",\n'
      '                   help="write final_model.pt at end of run, for "\n'
      '                        "analysis/selectivity.py. I/O only: excluded from "\n'
      '                        "config_hash in src/config.py")\n',
      marker='"--save_final_checkpoint"')

# (b2) train.py -- the save itself, immediately before the success finalize
SAVE = (
    '    if args.save_final_checkpoint:\n'
    '        torch.save({"state_dict": model.state_dict(),\n'
    '                    "width": args.width, "depth": args.depth,\n'
    '                    "act_fn": args.act_fn, "use_ln": use_ln,\n'
    '                    "dataset": args.dataset, "n_tasks": args.n_tasks,\n'
    '                    "seed": args.seed, "projection": args.projection,\n'
    '                    "config_hash": cfg["config_hash"],\n'
    '                    "schema_version": cfg["schema_version"]},\n'
    '                   os.path.join(args.run_dir, "final_model.pt"))\n\n')
ANCHOR = '    extra = {"n_rows": len(df), "n_steps": global_step,\n'
p = pathlib.Path("src/train.py"); s = p.read_text()
if 'torch.save({"state_dict"' in s:
    print("  src/train.py: save block already present")
else:
    assert s.count(ANCHOR) == 1, f"save anchor not unique: {s.count(ANCHOR)}"
    p.write_text(s.replace(ANCHOR, SAVE + ANCHOR))
    print("  src/train.py: save block inserted")

# (c) launch.py -- store_true flags must be declared or they are passed a value
p = pathlib.Path("src/launch.py"); s = p.read_text()
if "save_final_checkpoint" in s:
    print("  src/launch.py: already patched")
else:
    old = 'BOOL_FLAGS = {"track_drift", "log_grad_trace"}'
    assert s.count(old) == 1
    p.write_text(s.replace(old, 'BOOL_FLAGS = {"track_drift", "log_grad_trace", "save_final_checkpoint"}'))
    print("  src/launch.py: BOOL_FLAGS extended")

## 5. Verify the flag does not fork the comparability class

The property that matters: for a run's configuration, `config_hash` must be **identical**
whether `save_final_checkpoint` is absent, `True`, or `False`. If it is not, the launcher
marks every existing completed run `stale` and re-runs it, and `load_arm` refuses to combine
an old seed with a new one inside the same arm.

Tested against real configs already on disk. **If this fails, stop and fix the ignore set
before launching anything.**

(Note it is *not* valid to check the recorded `config_hash` of an old run against a
recomputation — those were written by an earlier code revision with a different argument set,
so they differ for reasons that have nothing to do with this patch.)

In [ ]:
import json, glob, sys
for m in [k for k in list(sys.modules) if k.startswith("src.")]:
    del sys.modules[m]
sys.path.insert(0, ".")
from src.config import config_hash

checked = 0
for c in sorted(glob.glob("experiments/permuted_mnist/*/*/seed_*/config.json"))[:20]:
    cfg = json.load(open(c))
    if cfg.get("status") != "complete":
        continue
    base = {k: v for k, v in cfg.items() if k != "config_hash"}
    absent = config_hash(base)
    on = config_hash({**base, "save_final_checkpoint": True})
    off = config_hash({**base, "save_final_checkpoint": False})
    assert absent == on == off, f"HASH FORKS on {c}: {absent} / {on} / {off}"
    checked += 1
print(f"config_hash is invariant to save_final_checkpoint on {checked} real configs")
assert checked > 0, "no completed runs found to check against -- verify by hand before launching"

## 6. The study: `S10_selectivity`

Four conditions x seeds 1,2,3. The figure has three panels — baseline, penalty, hard
projection — and `limit_ste` is run alongside because the repo is emphatic that `tangential`
and `ste` are **different algorithms and are never aliased**; having both means the panel
labelled "hard projection" can say which one it is.

Everything else is the canonical configuration, so these are directly comparable with
`S3_stiffness_curve`: `lambda_0.0000`, `lambda_0.0030` and `limit_tangential` there are the
same arms without checkpointing.

In [ ]:
import pathlib, sys

STUDY_SRC = '''

# ---------------------------------------------------------------- S10
# Checkpointed re-runs of the canonical conditions, for analysis/selectivity.py.
# Same configuration as the corresponding S3 arms; the only difference is that
# these write final_model.pt. save_final_checkpoint is I/O only and is excluded
# from config_hash (src/config.py), so it does not fork the comparability class.
study(
    "S10_selectivity",
    root="permuted_mnist/selectivity",
    question="Are hidden units task-shared or task-specific, and does the "
             "penalty move the network between those regimes?",
    claim="Supporting (mechanism, descriptive). Supplies the selectivity figure.",
    seeds=SEEDS_CURVE,
    phases=[{"name": "sweep", "arms": [
        {"dir": "baseline",
         "args": _canon(method="bp", lambda_rs=0.0, save_final_checkpoint=True)},
        {"dir": "penalty",
         "args": _canon(method="rs", lambda_rs=LAMBDA_STAR, save_final_checkpoint=True)},
        {"dir": "limit_tangential",
         "args": _canon(method="bp", lambda_rs=0.0, projection="tangential",
                        save_final_checkpoint=True)},
        {"dir": "limit_ste",
         "args": _canon(method="bp", lambda_rs=0.0, projection="ste",
                        save_final_checkpoint=True)},
    ]}],
)
'''

p = pathlib.Path("src/studies.py"); s = p.read_text()
if "S10_selectivity" in s:
    print("S10_selectivity already present")
else:
    p.write_text(s + STUDY_SRC); print("S10_selectivity appended")

for m in [k for k in list(sys.modules) if k.startswith("src.")]:
    del sys.modules[m]
from src.studies import STUDIES
st = STUDIES["S10_selectivity"]
print("arms:", [a["dir"] for a in st["phases"][0]["arms"]], "| seeds:", st["seeds"])

## 7. `analysis/selectivity.py`

For each hidden unit `i`, each task `k`, and each **layer separately**:

```
resp[i,k] = mean over that task's probe inputs of |post_activation_i|
sel[i,k]  = resp[i,k] / max_k resp[i,k]           # normalised per unit
```

Per-unit normalisation is what makes the heatmap answer the question rather than re-display
the unit's overall gain: without it a loud unit is a bright row whether or not it is
selective. Units dead on every task (`max` = 0) are dropped and counted, not divided by zero
— `dead_frac` in the main battery already tracks that population.

Two scalar summaries accompany the map, both per layer:

- **`selectivity_index`** = `1 - mean_k sel[i,k]`, per unit. 0 = responds equally to every
  task (fully shared); approaching 1 = responds to essentially one task (fully specific).
- **`participation_ratio`** = `(sum_k sel)^2 / sum_k sel^2`, the effective number of tasks a
  unit serves. Reported because the index alone cannot distinguish "two tasks" from "many
  tasks, unevenly".

Conventions honoured: per-layer, never averaged across layers; the probe is the first
`probe_size` **test** examples of each task, matching how `test_acc` is measured; the dataset
is reconstructed with the run's own seed, so the task sequence is the one the run actually saw.

In [ ]:
%%writefile analysis/selectivity.py
"""Per-neuron task selectivity from an end-of-run checkpoint.

Answers: are hidden units task-shared or task-specific, and does the radial
penalty move the network between those regimes?

For each layer separately -- per-layer metrics are never averaged across layers
(README, convention 3) -- and for each hidden unit i and task k:

    resp[i, k] = mean over task k's probe inputs of |post-activation_i|
    sel[i, k]  = resp[i, k] / max_k resp[i, k]

The per-unit normalisation is the point: an unnormalised map shows which units
are loud, not which units are selective.

Units dead on every task (max response exactly 0) cannot be normalised. They are
dropped and counted rather than silently becoming NaN rows.
"""
import argparse
import json
import os
import sys

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from src.models.mlp import MLP
from src.data.permuted_mnist import PermutedMNIST
from src.data.rotating_mnist import RotatingMNIST


def load_run(run_dir, device="cuda"):
    ckpt_path = os.path.join(run_dir, "final_model.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"{run_dir} has no final_model.pt -- it was run without "
            f"--save_final_checkpoint. Re-run that arm; the checkpoint cannot be "
            f"reconstructed after the fact.")
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = json.load(open(os.path.join(run_dir, "config.json")))
    if cfg.get("status") != "complete":
        raise ValueError(f"{run_dir} status={cfg.get('status')}, not reportable")
    act = nn.ReLU if ck["act_fn"] == "relu" else nn.LeakyReLU
    model = MLP(use_ln=ck["use_ln"], depth=ck["depth"], width=ck["width"],
                act_fn=act).to(device)
    model.load_state_dict(ck["state_dict"])
    model.eval()
    return model, ck, cfg


def responses(run_dir, probe_size=2000, tasks=None, device="cuda"):
    """(n_layers, units, n_tasks) mean absolute post-activation response."""
    model, ck, cfg = load_run(run_dir, device)
    ds_cls = {"permuted_mnist": PermutedMNIST,
              "rotating_mnist": RotatingMNIST}[ck["dataset"]]
    # Same seed as the run, so the task sequence is the one it actually saw.
    dataset = ds_cls(n_tasks=ck["n_tasks"], device=device, seed=ck["seed"])
    task_ids = list(range(ck["n_tasks"])) if tasks is None else list(tasks)
    fwd = {} if ck["projection"] in (None, "none") else {"projection": ck["projection"]}

    cols = []
    with torch.no_grad():
        for t in task_ids:
            _, _, x_test, _ = dataset.get_task_data(t)
            x = x_test[:probe_size].to(device)
            _, _, post = model(x, return_activations=True, **fwd)
            cols.append(np.stack([p.abs().mean(0).float().cpu().numpy() for p in post]))
    return np.stack(cols, axis=-1), ck, cfg


def selectivity(resp):
    """Per-unit-normalised map, index, participation ratio, dead count.

    resp: (units, n_tasks) for ONE layer."""
    mx = resp.max(axis=1)
    alive = mx > 0
    n_dead = int((~alive).sum())
    sel = resp[alive] / mx[alive][:, None]
    index = 1.0 - sel.mean(axis=1)
    s1, s2 = sel.sum(axis=1), (sel ** 2).sum(axis=1)
    pr = (s1 ** 2) / np.maximum(s2, 1e-12)
    return sel, index, pr, n_dead


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run_dir", required=True)
    ap.add_argument("--probe_size", type=int, default=2000)
    ap.add_argument("--every", type=int, default=1,
                    help="use every Nth task; 150 columns render fine, so the "
                         "default keeps all of them")
    ap.add_argument("--out", default=None, help="write the .npz here")
    a = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_tasks = json.load(open(os.path.join(a.run_dir, "config.json")))["n_tasks"]
    tasks = list(range(0, n_tasks, a.every))
    resp, ck, cfg = responses(a.run_dir, a.probe_size, tasks, device)

    out = {"tasks": np.array(tasks), "config_hash": cfg["config_hash"]}
    print(f"{a.run_dir}\n  config_hash {cfg['config_hash']}  seed {ck['seed']}  "
          f"{resp.shape[0]} layers x {resp.shape[1]} units x {resp.shape[2]} tasks")
    for L in range(resp.shape[0]):
        sel, index, pr, n_dead = selectivity(resp[L])
        out[f"resp_layer{L}"] = resp[L]
        out[f"sel_layer{L}"] = sel
        out[f"index_layer{L}"] = index
        out[f"pr_layer{L}"] = pr
        print(f"  layer {L}: selectivity_index {index.mean():.4f} +/- {index.std():.4f} | "
              f"participation_ratio {pr.mean():.2f} of {resp.shape[2]} tasks | "
              f"dead {n_dead}/{resp.shape[1]}")
    path = a.out or os.path.join(a.run_dir, "selectivity.npz")
    np.savez_compressed(path, **out)
    print("  ->", path)


if __name__ == "__main__":
    main()

## 8. Regression test

Every metric in this repo has one. This pins the two properties the figure depends on: a unit
responding equally to all tasks scores index 0 and participation ratio = n_tasks, and a unit
responding to exactly one task scores index -> 1 and participation ratio 1. It also pins that
per-unit normalisation removes overall gain, which is the reason the normalisation is there.

In [ ]:
%%writefile tests/test_selectivity.py
import numpy as np

from analysis.selectivity import selectivity


def test_shared_unit_scores_zero_and_full_participation():
    sel, index, pr, n_dead = selectivity(np.ones((1, 10)))
    assert np.isclose(index[0], 0.0)
    assert np.isclose(pr[0], 10.0)
    assert n_dead == 0


def test_task_specific_unit_scores_near_one_and_participation_one():
    resp = np.zeros((1, 10)); resp[0, 3] = 1.0
    sel, index, pr, _ = selectivity(resp)
    assert np.isclose(index[0], 0.9)
    assert np.isclose(pr[0], 1.0)


def test_normalisation_removes_overall_gain():
    """A loud unit and a quiet unit with the same shape must score identically:
    the map shows selectivity, not gain."""
    shape = np.array([1.0, 0.5, 0.25, 0.1])
    sel, index, pr, _ = selectivity(np.stack([shape, 100 * shape]))
    assert np.allclose(sel[0], sel[1])
    assert np.isclose(index[0], index[1])


def test_dead_units_are_dropped_not_nan():
    resp = np.zeros((3, 5)); resp[0] = 1.0; resp[2] = 1.0
    sel, index, pr, n_dead = selectivity(resp)
    assert n_dead == 1
    assert sel.shape[0] == 2
    assert np.isfinite(index).all() and np.isfinite(pr).all()

In [ ]:
!PYTHONPATH=. python3 -m pytest tests/ -q

## 10. Carry over earlier sessions

Runs finished in an earlier session are copied back from every attached input dataset, along
with their checkpoints. A run left at `status: running` by an interrupted session is treated as
failed and re-run.

In [ ]:
import glob, os, shutil

restored = 0
for src in glob.glob(f"/kaggle/input/*/rs_for_optimisation/{ROOT}"):
    for run in glob.glob(os.path.join(src, "*", "seed_*")):
        dst = os.path.join(WORK, ROOT, os.path.relpath(run, src))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copytree(run, dst)
            restored += 1
print(f"restored {restored} run dir(s) from {len(glob.glob('/kaggle/input/*'))} attached input(s)")

## 11. Resolve the shard

`--only` matches substrings against each run directory, and arm dirs are `<condition>/seed_N`,
so a condition name selects its 3 runs. `"auto"` picks the condition with the most work left,
so an unattended session always does something useful.

In [ ]:
import glob, json, os

def completed(cond):
    n = 0
    for c in glob.glob(os.path.join(ROOT, cond, "seed_*", "config.json")):
        try:
            if json.load(open(c)).get("status") == "complete":
                n += 1
        except Exception:
            pass
    return n

counts = {c: completed(c) for c in CONDITIONS}
print("completed per condition (of 3 each):", counts)

if SHARD == "auto":
    remaining = {c: 3 - n for c, n in counts.items() if n < 3}
    if not remaining:
        SHARD_RESOLVED, ONLY = None, None
        print("\nAll 12 runs are complete -- nothing to launch. Skip to the analysis cells.")
    else:
        SHARD_RESOLVED = max(remaining, key=lambda c: (remaining[c], -CONDITIONS.index(c)))
        ONLY = f"{SHARD_RESOLVED}/"
        print(f"\nauto -> {SHARD_RESOLVED}  ({remaining[SHARD_RESOLVED]} runs left)")
elif SHARD == "all":
    SHARD_RESOLVED, ONLY = "all", None
    print("\nrunning ALL 12 -- expect ~9 h")
else:
    SHARD_RESOLVED = SHARD
    ONLY = ",".join(s.strip() + "/" for s in SHARD.split(","))
    print(f"\nexplicit -> {SHARD_RESOLVED}  (--only {ONLY})")

## 12. Dry run

12 runs. Check that `--save_final_checkpoint` appears as a **bare flag** and not as
`--save_final_checkpoint True` — that is what the `BOOL_FLAGS` edit was for, and getting it
wrong kills every run in the first second.

In [ ]:
!PYTHONPATH=. python3 -m src.launch --study {STUDY} --dry-run

## 9. Launch — the overnight cell

Guards in force:

* **Time budget.** At `TIME_BUDGET_H` the launcher gets SIGTERM, then SIGKILL 60 s later. The
  notebook continues to the analysis cells so `/kaggle/working` is saved. Runs in flight are
  lost and re-run next session; runs already finished are safe.
* **Heartbeat.** A progress line every 10 minutes with elapsed time and completed count, so a
  stalled session is visible in the log rather than looking like a slow one.
* **Output throttling.** tqdm redraws are dropped; only real events are printed. A 4.5 h run
  otherwise produces tens of MB of progress bars.
* **Retries.** The launcher re-attempts a failed run twice (`--retries 2`) — a shared GPU can
  OOM-kill a run through no fault of its own — and records failures in `_launch_report.json`.
* **No exception escapes.** A crash here is caught and reported so the later cells still run.

In [ ]:
import glob, json, os, signal, subprocess, sys, threading, time

def n_complete():
    n = 0
    for c in glob.glob(os.path.join(ROOT, "*", "seed_*", "config.json")):
        try:
            if json.load(open(c)).get("status") == "complete":
                n += 1
        except Exception:
            pass
    return n

if SHARD_RESOLVED is None:
    print("nothing to launch")
else:
    cmd = [sys.executable, "-u", "-m", "src.launch", "--study", STUDY,
           "--gpus", GPUS, "--concurrency", str(CONCURRENCY),
           "--threads", str(THREADS), "--retries", "2"]
    if ONLY:
        cmd += ["--only", ONLY]
    print(" ".join(cmd), flush=True)

    deadline = SESSION_START + TIME_BUDGET_H * 3600
    env = {**os.environ, "PYTHONPATH": ".", "PYTHONUNBUFFERED": "1"}
    t0 = time.time()
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, preexec_fn=os.setsid)

    state = {"stop": False}

    def watchdog():
        """Enforce the deadline and beat independently of the child's output.

        The read loop below blocks on p.stdout, so a run that goes quiet -- a hang,
        a wedged GPU -- would never reach a deadline check placed in that loop.
        This thread owns both the budget and the heartbeat for that reason.
        """
        last_beat = time.time()
        while not state["stop"]:
            time.sleep(5)
            now = time.time()
            if now - last_beat >= 600:
                print(f"    [heartbeat] {(now - t0) / 3600:.2f} h elapsed, "
                      f"{n_complete()}/12 complete", flush=True)
                last_beat = now
            if now > deadline and p.poll() is None:
                print(f"\n!! TIME BUDGET {TIME_BUDGET_H} h REACHED -- stopping the launcher "
                      f"so this session's finished runs are saved.", flush=True)
                state["stop"] = True
                try:
                    os.killpg(os.getpgid(p.pid), signal.SIGTERM)
                    time.sleep(60)
                    if p.poll() is None:
                        os.killpg(os.getpgid(p.pid), signal.SIGKILL)
                except ProcessLookupError:
                    pass
                return

    wd = threading.Thread(target=watchdog, daemon=True)
    wd.start()
    try:
        for line in p.stdout:
            s = line.rstrip("\n")
            # tqdm redraws would flood the log; drop them, keep real events
            if "\r" in line or ("%|" in s and s.strip().endswith("]")):
                continue
            if s.strip():
                print(s, flush=True)
    except Exception as e:
        print("launch loop error:", repr(e))
    finally:
        state["stop"] = True
    try:
        p.wait(timeout=180)
    except Exception:
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception:
            pass
    print(f"\nexit {p.returncode} after {(time.time() - t0) / 3600:.2f} h; "
          f"{n_complete()}/12 runs complete")

## 14. Run status and checkpoints on disk

In [ ]:
import glob, json, os
import pandas as pd

try:
    rows = []
    for c in sorted(glob.glob(os.path.join(ROOT, "*", "seed_*", "config.json"))):
        cfg = json.load(open(c)); d = os.path.dirname(c)
        ck = os.path.join(d, "final_model.pt")
        rows.append({"arm": c.split("/")[-3], "seed": cfg.get("seed"),
                     "status": cfg.get("status", "?"),
                     "method": cfg.get("method"), "projection": cfg.get("projection"),
                     "lambda_rs": cfg.get("lambda_rs"),
                     "hours": round(cfg.get("wall_clock_sec", 0) / 3600, 2),
                     "ckpt_MB": round(os.path.getsize(ck) / 1e6, 1) if os.path.exists(ck) else None,
                     "config_hash": cfg.get("config_hash")})
    df = pd.DataFrame(rows)
    if len(df):
        print(df.sort_values(["arm", "seed"]).to_string(index=False))
        print("\ncomplete:", int((df.status == "complete").sum()), "of 12")
        print("checkpoints on disk:", int(df.ckpt_MB.notna().sum()))
        missing = df[(df.status == "complete") & (df.ckpt_MB.isna())]
        if len(missing):
            print("!! complete but NO checkpoint -- these ran before the patch and must be "
                  "re-run with --force:\n", missing[["arm", "seed"]].to_string(index=False))
    else:
        print("no runs yet")
    rep = os.path.join(ROOT, "_launch_report.json")
    if os.path.exists(rep):
        print("\n_launch_report.json:", json.dumps(json.load(open(rep)))[:1500])
except Exception as e:
    print("status cell error (non-fatal):", repr(e))

## 15. Compute selectivity

Per run, all layers, saved next to the checkpoint as `selectivity.npz`. Skips runs already
processed so a re-run of the notebook is cheap.

In [ ]:
import glob, os, subprocess, sys

try:
    for d in sorted(glob.glob(os.path.join(ROOT, "*", "seed_*"))):
        if not os.path.exists(os.path.join(d, "final_model.pt")):
            continue
        if os.path.exists(os.path.join(d, "selectivity.npz")):
            print("already done:", d); continue
        r = subprocess.run([sys.executable, "-m", "analysis.selectivity", "--run_dir", d],
                           env={**os.environ, "PYTHONPATH": "."},
                           capture_output=True, text=True)
        print(r.stdout.strip() or r.stderr.strip()[-500:])
except Exception as e:
    print("selectivity error (non-fatal):", repr(e))

## 14. The figure

Units x tasks heatmap, one panel per condition, layer 0 by default. Follows the conventions
already in `analysis/figures.py`: serif 7pt, ink-coloured text rather than series colours, no
dual axes, recessive grid. A sequential colormap is correct here — the quantity is normalised
response in [0, 1] with a meaningful zero, so it is not diverging data and must not get a
diverging map.

Rows are sorted by each unit's preferred task, which is what turns a noise-looking block into
readable structure: a task-specific network shows a diagonal band, a task-shared network shows
uniform brightness. The sort is presentational — the index and participation ratio in the
panel titles are the quantitative claim, and they are sort-invariant.

In [ ]:
%%writefile -a analysis/figures.py


# ------------------------------------------------------- per-neuron selectivity
def fig_selectivity(layer=0, seed=1, n_units=300, conditions=None):
    """Units x tasks normalised-response heatmap, one panel per condition.

    Rows are sorted by preferred task, so a task-specific network reads as a
    diagonal band and a task-shared one as uniform brightness. The sort is
    presentational; the selectivity index and participation ratio in the panel
    titles are the quantitative claim and do not depend on it.

    Sequential colormap, not diverging: the quantity is normalised response in
    [0, 1] with a meaningful zero.
    """
    from matplotlib.colors import Normalize

    SEL = os.path.join(EXP, "permuted_mnist/selectivity")
    conditions = conditions or [("baseline", "baseline"),
                                ("penalty", "penalty (lambda*)"),
                                ("limit_tangential", "hard projection (tangential)")]
    panels = []
    for d, label in conditions:
        f = os.path.join(SEL, d, f"seed_{seed}", "selectivity.npz")
        if not os.path.exists(f):
            print(f"  [missing] {f}")
            continue
        z = np.load(f)
        panels.append((label, z[f"sel_layer{layer}"], z[f"index_layer{layer}"],
                       z[f"pr_layer{layer}"], z["tasks"]))
    if not panels:
        print("fig_selectivity: no selectivity.npz found; run analysis/selectivity.py first")
        return

    fig, axes = plt.subplots(1, len(panels), figsize=(2.3 * len(panels), 2.5),
                             constrained_layout=True)
    axes = np.atleast_1d(axes)
    norm = Normalize(0, 1)
    for ax, (label, sel, index, pr, tasks) in zip(axes, panels):
        order = np.argsort(sel.argmax(axis=1))
        show = sel[order][:n_units]
        im = ax.imshow(show, aspect="auto", cmap="magma", norm=norm,
                       interpolation="nearest",
                       extent=[tasks[0], tasks[-1], show.shape[0], 0])
        ax.set_title(f"{label}\nindex {index.mean():.3f}   PR {pr.mean():.1f}",
                     color=INK, pad=3)
        ax.set_xlabel("task")
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    axes[0].set_ylabel(f"hidden unit (layer {layer}, sorted)")
    cb = fig.colorbar(im, ax=list(axes), fraction=0.025, pad=0.01)
    cb.set_label("response, normalised per unit", color=INK)
    cb.outline.set_visible(False)
    os.makedirs(OUT, exist_ok=True)
    path = os.path.join(OUT, f"selectivity_layer{layer}.pdf")
    fig.savefig(path)
    fig.savefig(path.replace(".pdf", ".png"))
    plt.close(fig)
    print("wrote", path)

In [ ]:
import os, sys
try:
    for m in [k for k in list(sys.modules) if k.startswith(("analysis", "src"))]:
        del sys.modules[m]
    sys.path.insert(0, ".")
    import analysis.figures as F
    F.fig_selectivity(layer=0, seed=1)
    from IPython.display import Image, display
    p = os.path.join(F.OUT, "selectivity_layer0.png")
    if os.path.exists(p):
        display(Image(p))
except Exception as e:
    print("figure error (non-fatal):", repr(e))

**Look at the rendered image before calling this done** — that is in the task description, not
a formality. Check: is the colour range actually used, or is everything saturated at one end?
Are the labels legible at 7pt? Does the sort reveal structure or manufacture it?

Layers 1 and 2 for the appendix — per layer, never collapsed into one number:

In [ ]:
try:
    for L in (1, 2):
        F.fig_selectivity(layer=L, seed=1)
except Exception as e:
    print("figure error (non-fatal):", repr(e))

## 15. Quantitative summary

The figure shows the structure; this is the number that goes in the text. Paired across seeds
where they align, with test statistic, p, n and sign split (convention 4).

In [ ]:
try:
    import glob, numpy as np, pandas as pd
    from scipy import stats

    rows = []
    for f in sorted(glob.glob("experiments/permuted_mnist/selectivity/*/seed_*/selectivity.npz")):
        arm, seed = f.split("/")[-3], int(f.split("/")[-2].split("_")[1])
        z = np.load(f)
        for L in range(3):
            if f"index_layer{L}" not in z:
                continue
            rows.append({"arm": arm, "seed": seed, "layer": L,
                         "selectivity_index": float(z[f"index_layer{L}"].mean()),
                         "participation_ratio": float(z[f"pr_layer{L}"].mean()),
                         "n_units": int(z[f"index_layer{L}"].shape[0])})
    d = pd.DataFrame(rows)
    if not len(d):
        print("no selectivity.npz yet")
    else:
        print(d.groupby(["layer", "arm"])[["selectivity_index", "participation_ratio", "n_units"]]
               .agg(["mean", "std", "count"]).round(4).to_string())
        print()
        for L in sorted(d.layer.unique()):
            b = d[(d.arm == "baseline") & (d.layer == L)].set_index("seed")["selectivity_index"]
            for arm in [a for a in sorted(d.arm.unique()) if a != "baseline"]:
                o = d[(d.arm == arm) & (d.layer == L)].set_index("seed")["selectivity_index"]
                common = b.index.intersection(o.index)
                if len(common) < 2:
                    print(f"layer {L}  {arm} vs baseline: n={len(common)}, not reportable")
                    continue
                diff = o[common] - b[common]
                t, pv = stats.ttest_rel(o[common], b[common])
                print(f"layer {L}  {arm} vs baseline: delta={diff.mean():+.4f}  t={t:.3f}  "
                      f"p={pv:.4f}  n={len(common)}  "
                      f"sign {int((diff > 0).sum())}+/{int((diff < 0).sum())}-")

except Exception as e:
    print("summary error (non-fatal):", repr(e))

## 18. Refresh STUDY.md and save the output

Everything under `/kaggle/working` becomes this notebook's output. MNIST is deleted first;
checkpoints are kept — they are the artifact the analysis depends on, and re-generating one
costs another 90 minutes.

In [ ]:
import os, shutil
try:
    !PYTHONPATH=. python3 -m src.docs
    shutil.rmtree(os.path.join(WORK, "data"), ignore_errors=True)
    !du -sh {WORK}/{ROOT}
    !du -sh /kaggle/working
    print(open(os.path.join(ROOT, "STUDY.md")).read()[:2000])
except Exception as e:
    print("docs/cleanup error (non-fatal):", repr(e))

## When you are done

The code written here lives only in the Kaggle working copy. To land it, carry these back to a
clone with push access:

* `src/config.py` — one line in the ignore set, **the load-bearing edit**
* `src/train.py` — flag + save block
* `src/launch.py` — `BOOL_FLAGS`
* `src/studies.py` — `S10_selectivity`
* `analysis/selectivity.py`, `tests/test_selectivity.py`, `analysis/figures.py`

Then write the finding into `experiments/permuted_mnist/selectivity/STUDY.md` between the
`<!-- FINDING -->` markers, saying which layers the claim holds at and at what n.

**Look at the rendered figure before calling it done** — that is in the task description, not a
formality. Is the colour range used, or is everything saturated at one end? Does the sort
reveal structure or manufacture it?

**Report it honestly if the map is boring.** "Units are task-shared in every condition and the
penalty does not change that" is a real result about the mechanism. The repo's whole posture —
six conventions written after four rounds had to be withdrawn — is that a null gets reported as
a null.

## Trim the output

`/kaggle/working` becomes this notebook's output dataset, and it currently holds the entire
cloned repo — ~98 MB, almost all of it the archived `experiments/` tree that came from git and
is identical in every session. Attaching that to the next session is slow for no benefit.

This keeps only what the next session actually needs: **this study's run directories** (their
`config.json`, `metrics.parquet`, checkpoints and `selectivity.npz`), the study's `STUDY.md` and
`_launch_report.json`, and any generated figures. Everything else is re-cloned from git next
time.

It runs **last**, after `src.docs`, and never deletes the working directory itself. The keep
list is printed before anything is removed, and a guard refuses to run if the study directory
is missing — so a bug here cannot silently throw away a night of compute.

In [ ]:
import os, shutil

try:
    root = "/kaggle/working"
    repo = os.path.join(root, "rs_for_optimisation")
    keep_rel = [ROOT, "latex/figs"]                    # study runs, and any generated figures

    keep_abs = [os.path.join(repo, r) for r in keep_rel]
    study_dir = os.path.join(repo, ROOT)
    if not os.path.isdir(study_dir):
        print(f"REFUSING to trim: {study_dir} does not exist")
    else:
        n_runs = sum(1 for _, ds, fs in os.walk(study_dir) if "config.json" in fs)
        before = sum(os.path.getsize(os.path.join(dp, f))
                     for dp, _, fs in os.walk(root) for f in fs
                     if os.path.exists(os.path.join(dp, f)))
        print(f"keeping {n_runs} run dir(s) under {ROOT}")

        # Everything under the repo that is not on a kept path, plus stray top-level items.
        def protected(path):
            return any(path == k or path.startswith(k + os.sep) or k.startswith(path + os.sep)
                       for k in keep_abs)

        removed = 0
        for dirpath, dirnames, filenames in os.walk(repo, topdown=True):
            if protected(dirpath):
                if dirpath in keep_abs:
                    dirnames[:] = []                   # keep this subtree whole
                continue
            for d in list(dirnames):
                full = os.path.join(dirpath, d)
                if not protected(full):
                    shutil.rmtree(full, ignore_errors=True)
                    dirnames.remove(d)
                    removed += 1
            for f in filenames:
                full = os.path.join(dirpath, f)
                if not protected(full):
                    try:
                        os.remove(full); removed += 1
                    except OSError:
                        pass

        after = sum(os.path.getsize(os.path.join(dp, f))
                    for dp, _, fs in os.walk(root) for f in fs
                    if os.path.exists(os.path.join(dp, f)))
        kept_runs = sum(1 for _, ds, fs in os.walk(study_dir) if "config.json" in fs)
        print(f"removed {removed} item(s): {before/1e6:.1f} MB -> {after/1e6:.1f} MB")
        print(f"run dirs still present: {kept_runs} (was {n_runs})")
        assert kept_runs == n_runs, "TRIM BUG: run directories were lost"
        for dp, _, fs in os.walk(root):
            if fs:
                print(" ", os.path.relpath(dp, root), f"({len(fs)} files)")
except Exception as e:
    print("trim error (non-fatal):", repr(e))